# 23장 생성모델과 언어모델 — 다음 글자를 맞히면 글을 쓸 수 있다

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch23_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `p01_bigram.py` 와 `charlm.py` 를 치고 실행(`STEPS=500` 으로 먼저) | [ch23_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch23_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `CharLM.forward` 와 `generate` 각 줄에 shape과 "왜 마지막 위치만"을 주석으로 | [ch23_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch23_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `generate` 복원. 그다음 바이그램 확률표로 같은 `generate` 인터페이스를 만들어 두 모델을 같은 프롬프트로 비교 | [ch23_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch23_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "23"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/23")

준비 끝. 작업 폴더: code/23


## `p01_bigram.py` — 23.1 가장 작은 언어 모델: 바이그램. "이 글자 다음에 어떤 글자가 얼마나 자주 오는가" 를 세기만 한다.

In [ ]:
# 23.1 가장 작은 언어 모델: 바이그램. "이 글자 다음에 어떤 글자가 얼마나 자주 오는가" 를 세기만 한다.
import os
import math
import torch

DATA = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "statutes", "민법.txt")
text = open(DATA, encoding="utf-8").read()
itos = sorted(set(text))
stoi = {c: i for i, c in enumerate(itos)}
V = len(itos)
print(f"글자 {len(text):,} 개, 글자 종류 {V}")

글자 147,324 개, 글자 종류 510


In [ ]:
# 세기: counts[a, b] = a 다음에 b 가 온 횟수
counts = torch.zeros(V, V)
ids = [stoi[c] for c in text]
for a, b in zip(ids, ids[1:]):
    counts[a, b] += 1

In [ ]:
# 확률: 행마다 합이 1 이 되게. +1 은 한 번도 안 나온 쌍에 아주 작은 확률을 주기 위해(스무딩)
probs = (counts + 1) / (counts + 1).sum(1, keepdim=True)

In [ ]:
# '채' 다음에 가장 잘 오는 글자
a = stoi["채"]
top = probs[a].topk(5)
print("'채' 다음:", [(itos[i], round(p, 3)) for p, i in zip(top.values.tolist(), top.indices.tolist())])

'채' 다음: [('무', 0.33), ('권', 0.288), ('염', 0.004), ('취', 0.003), ('변', 0.002)]


In [ ]:
# 생성: 확률대로 다음 글자를 뽑아 이어 붙인다
g = torch.Generator().manual_seed(0)
cur = stoi["제"]
out = ["제"]
for _ in range(80):
    cur = torch.multinomial(probs[cur], 1, generator=g).item()
    out.append(itos[cur])
print("바이그램 생성:", "".join(out))

바이그램 생성: 제춘조제언증맺起]육력측月탁算所年질살밟불희괴절천친②12조의하며밖償還母된 않數몽惡몽갈죄姻직장의 것을 후색요예척초력(系무를 기에 발할 언벌월나送터 통항


In [ ]:
# 얼마나 놀랍지 않은가: 평균 -log p (교차 엔트로피) 와 perplexity
nll = -sum(math.log(probs[a, b].item()) for a, b in zip(ids[:20000], ids[1:20001])) / 20000
print(f"앞 2만 글자의 평균 손실 {nll:.3f}  perplexity {math.exp(nll):.1f}  (무작위 추측이면 {V})")

앞 2만 글자의 평균 손실 2.904  perplexity 18.2  (무작위 추측이면 510)


## `charlm.py` — 23장 문자 단위 언어 모델: "다음 글자는 무엇인가" 를 배우면 문장을 만들어 낼 수 있다.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 23장 문자 단위 언어 모델: "다음 글자는 무엇인가" 를 배우면 문장을 만들어 낼 수 있다.
# 데이터: 민법 조문 전체(약 19만 자). 모델: 22장의 Transformer 블록을 '앞만 보는' 마스크로 쓴 디코더.
import math
import os
import sys
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "22"))
from transformer import PositionalEncoding, MultiHeadAttention

DATA = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "statutes", "민법.txt")


# ---- 1. 데이터: 글자 하나가 토큰. 어휘는 등장한 글자 전부
def load_text():
    with open(DATA, encoding="utf-8") as f:
        return f.read()


class CharData:
    def __init__(self, text, block=64, split=0.9):
        self.itos = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.itos)}
        ids = torch.tensor([self.stoi[c] for c in text], dtype=torch.long)
        n = int(len(ids) * split)
        self.train, self.val = ids[:n], ids[n:]
        self.block = block

    def encode(self, s):
        return torch.tensor([self.stoi.get(c, 0) for c in s], dtype=torch.long)

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)

    def batch(self, split, B, g):
        """무작위 위치 B 개에서 block 길이의 (입력, 정답) 을 잘라 낸다. 정답 = 입력을 한 글자 밀어 놓은 것."""
        src = self.train if split == "train" else self.val
        ix = torch.randint(len(src) - self.block - 1, (B,), generator=g)
        x = torch.stack([src[i: i + self.block] for i in ix])
        y = torch.stack([src[i + 1: i + self.block + 1] for i in ix])
        return x, y


# ---- 2. 모델: 인코더 블록과 같되, 어텐션이 미래 글자를 보지 못하게 마스크
class CausalBlock(nn.Module):
    def __init__(self, d, h, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d, h)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d))
        self.n1, self.n2, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(dropout)

    def forward(self, x, causal_mask):
        B, L, d = x.shape
        q = k = v = self.n1(x)
        # MultiHeadAttention 은 pad_mask 만 받으므로 여기서 직접 계산한다
        a = self.attn
        q = a.wq(q).view(B, L, a.h, a.dk).transpose(1, 2)
        k = a.wk(k).view(B, L, a.h, a.dk).transpose(1, 2)
        v = a.wv(v).view(B, L, a.h, a.dk).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1) / math.sqrt(a.dk)).masked_fill(causal_mask[:L, :L], float("-inf"))
        out = (F.softmax(scores, -1) @ v).transpose(1, 2).reshape(B, L, d)
        x = x + self.drop(a.wo(out))
        x = x + self.drop(self.ff(self.n2(x)))
        return x


class CharLM(nn.Module):
    def __init__(self, vocab, block=64, d=128, h=4, d_ff=256, n_layers=3):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = PositionalEncoding(d, block)
        self.blocks = nn.ModuleList([CausalBlock(d, h, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)                          # 각 위치에서 "다음 글자" 의 점수 vocab 개
        mask = torch.triu(torch.ones(block, block, dtype=torch.bool), diagonal=1)   # 위 삼각 = 미래
        self.register_buffer("mask", mask)
        self.block = block

    def forward(self, ids):                                       # [B, L] → [B, L, vocab]
        x = self.pos(self.emb(ids) * math.sqrt(self.emb.embedding_dim))
        for blk in self.blocks:
            x = blk(x, self.mask)
        return self.head(self.norm(x))

    @torch.no_grad()
    def generate(self, ids, n_new, temperature=1.0, g=None):
        """앞 글자들을 주면 다음 글자를 뽑고, 그것을 다시 입력에 붙여 반복한다 (자기회귀)."""
        for _ in range(n_new):
            ctx = ids[:, -self.block:]
            logits = self(ctx)[:, -1] / temperature                # 마지막 위치의 점수만
            probs = F.softmax(logits, -1)
            nxt = torch.multinomial(probs, 1, generator=g)         # 확률대로 하나 뽑기
            ids = torch.cat([ids, nxt], 1)
        return ids


if __name__ == "__main__":
    torch.manual_seed(0)
    g = torch.Generator().manual_seed(0)
    data = CharData(load_text(), block=64)
    print(f"글자 수 {len(data.train) + len(data.val):,} | 어휘(글자 종류) {len(data.itos)} | 학습 {len(data.train):,} 검증 {len(data.val):,}")
    x, y = data.batch("train", 2, g)
    print("입력:", repr(data.decode(x[0][:20].tolist())), "\n정답:", repr(data.decode(y[0][:20].tolist())), "(한 글자 밀림)")

    model = CharLM(len(data.itos))
    print(f"파라미터 {sum(p.numel() for p in model.parameters()):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    steps = int(os.environ.get("STEPS", 1500))
    t0 = time.time()
    for step in range(1, steps + 1):
        x, y = data.batch("train", 32, g)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))   # 모든 위치에서 다음 글자 맞히기
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if step % 250 == 0 or step == 1:
            model.eval()
            with torch.no_grad():
                vx, vy = data.batch("val", 64, g)
                vl = F.cross_entropy(model(vx).reshape(-1, logits.size(-1)), vy.reshape(-1)).item()
            model.train()
            print(f"걸음 {step:5d}  학습 손실 {loss.item():.3f}  검증 손실 {vl:.3f}  (검증 perplexity {math.exp(vl):.1f})  {time.time() - t0:.0f}초")
    model.eval()
    for prompt in ["제1조(", "채무자가 ", "①"]:
        ids = data.encode(prompt).unsqueeze(0)
        out = model.generate(ids, 120, temperature=0.8, g=g)
        print(f"\n[{prompt}] → {data.decode(out[0].tolist())}")
```

먼저 500걸음만 돌려 봅니다.

In [ ]:
import os
os.environ["STEPS"] = "500"
!python charlm.py

글자 수 147,324 | 어휘(글자 종류) 510 | 학습 132,591 검증 14,733
입력: ' 가지를 제거할 수 있다. ③인접지의' 
정답: '가지를 제거할 수 있다. ③인접지의 ' (한 글자 밀림)
파라미터 528,766
걸음     1  학습 손실 6.389  검증 손실 6.389  (검증 perplexity 595.1)  0초
걸음   250  학습 손실 2.959  검증 손실 3.055  (검증 perplexity 21.2)  24초
걸음   500  학습 손실 2.607  검증 손실 2.736  (검증 perplexity 15.4)  46초

[제1조(] → 제1조(피의 일의 못한 목적으로 규정대리권의 부재산인한 효 있는 배상 받은 의 ①금의 법원은 본정후견인이행권을 선택자의 성부동의 법원은 지의 목적으로 제하여 친성년후견의하여 권의 사중하지 성년후견인이 있다.
제8조(시나 ②

[채무자가 ] → 채무자가 ④하여 미성년후견인이 따른 약자의 ①복하고 무자가 의 물를 또는 추인의 가 후 있다. 득담보존속에 부매의 반한다. 성년을 금를 부한 쌍하할 이사의 대방자가, ②전항의 이 채권에 제1259조(비용한 재여 그러 채무으로

[①] → ① 수을 양의한 때에는 수 취소멸한다.
제4조(청구사개시 증대하지령에는 충으로 ②전세권을 손해 친권자에게 때에는 권자의 관한 규정법원의 모는 , 당권) 기간무자가정한 알 법원은 취동의 전세권약수 수를 전세권자가 이행위


이제 책과 같은 1500걸음입니다. CPU 에서는 몇 분 걸립니다.

In [ ]:
os.environ["STEPS"] = "1500"
!python charlm.py

글자 수 147,324 | 어휘(글자 종류) 510 | 학습 132,591 검증 14,733
입력: ' 가지를 제거할 수 있다. ③인접지의' 
정답: '가지를 제거할 수 있다. ③인접지의 ' (한 글자 밀림)
파라미터 528,766
걸음     1  학습 손실 6.389  검증 손실 6.389  (검증 perplexity 595.1)  0초
걸음   250  학습 손실 2.959  검증 손실 3.055  (검증 perplexity 21.2)  19초
걸음   500  학습 손실 2.607  검증 손실 2.736  (검증 perplexity 15.4)  39초
걸음   750  학습 손실 2.429  검증 손실 2.699  (검증 perplexity 14.9)  58초
걸음  1000  학습 손실 2.413  검증 손실 2.471  (검증 perplexity 11.8)  78초
걸음  1250  학습 손실 2.360  검증 손실 2.438  (검증 perplexity 11.4)  97초
걸음  1500  학습 손실 2.284  검증 손실 2.440  (검증 perplexity 11.5)  117초

[제1조(] → 제1조(경우에 규정) 제삼자가 제삼자가 있는 경우에는 것으로 다. 다.
제770조(준용규정법률행위에 의 의 및 부의 ④ 인이 법원의 점) 경우에 각자가 권리, 동상의) 처분, 의사를 양수 소멸시효력을 위하여 지 위하여 때에

[채무자가 ] → 채무자가 대금의 채무의 촌 상권으로 대방이자가 변제공하지 못한다. 그 그 배서로 부는 그 능력자가 지 거절한 못한다.
제7조(除이행자가 잃는 요역권을 제의 취소유자에 그 인에 선식반환을 위를 청구하여 설정하여 못한다.
제31

[①] → ①의 무를 하여 그 소비용하여는 및 사유가로 ②대방이에 상을 법원의 청구할 수 있다. ③권권의 때에 기타인하지 때에 의 이의무자있다. 44조의 전조제한 정은 규정) 전세권 제12조의 그 규정은 제었고지상당권이 제90조


## 다음 단계

- 2단계 「주석 달기」: [ch23_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch23_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch23_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch23_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.